# Lesson 15: Support Vector Machines

*ML Academy — companion notebook*

Maximum margins, the hinge loss, and the kernel trick for curved boundaries.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-15.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 9. In Python

In [1]:
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = make_circles(n_samples=400, noise=0.2, factor=0.5, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)

for kernel in ["linear", "poly", "rbf"]:
    m = make_pipeline(StandardScaler(), SVC(kernel=kernel, degree=2)).fit(X_train, y_train)
    print(f"{kernel:6s} test accuracy {m.score(X_test, y_test):.3f}   support vectors {m[-1].n_support_.sum()}")

grid = {"svc__C": [0.1, 1, 10, 100], "svc__gamma": [0.01, 0.1, 1, 10]}
search = GridSearchCV(make_pipeline(StandardScaler(), SVC(kernel="rbf")), grid, cv=5).fit(X_train, y_train)
print("best:", search.best_params_, "CV", round(search.best_score_, 3), "test", round(search.score(X_test, y_test), 3))

m = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=100, gamma=10)).fit(X_train, y_train)
print("C=100, gamma=10: train", round(m.score(X_train, y_train), 3), "test", round(m.score(X_test, y_test), 3))

linear test accuracy 0.470   support vectors 297
poly   test accuracy 0.950   support vectors 97
rbf    test accuracy 0.930   support vectors 112


best: {'svc__C': 0.1, 'svc__gamma': 1} CV 0.893 test 0.9
C=100, gamma=10: train 0.983 test 0.86


## 12. Going deeper: the dual problem, kernels and hinge loss

In [2]:
import numpy as np

# The kernel trick: (x·z + 1)² equals the dot product of explicit degree-2 features
def phi(v):
    a, b = v
    return np.array([1, np.sqrt(2) * a, np.sqrt(2) * b, a * a, b * b, np.sqrt(2) * a * b])

x, z = np.array([1.0, 2.0]), np.array([3.0, -1.0])
print("kernel (x·z + 1)²:     ", (x @ z + 1) ** 2)
print("explicit φ(x)·φ(z):    ", round(phi(x) @ phi(z), 10))

kernel (x·z + 1)²:      4.0
explicit φ(x)·φ(z):     4.0


In [3]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

X, y = make_moons(n_samples=400, noise=0.3, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, random_state=0)
print("    C   gamma   support vectors   train   test")
for C in [0.1, 1, 100]:
    for gamma in [0.1, 1, 30]:
        m = SVC(C=C, gamma=gamma).fit(Xtr, ytr)
        print(f"{C:5} {gamma:7}   {m.n_support_.sum():15d}   {m.score(Xtr, ytr):5.3f}   {m.score(Xte, yte):5.3f}")

    C   gamma   support vectors   train   test
  0.1     0.1               158   0.810   0.860
  0.1       1               127   0.885   0.895
  0.1      30               193   0.535   0.470
    1     0.1                95   0.860   0.870
    1       1                69   0.905   0.910
    1      30               158   0.915   0.900
  100     0.1                64   0.900   0.875
  100       1                52   0.905   0.885
  100      30               118   1.000   0.845


In [4]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

# Hinge loss and log loss give very similar linear boundaries
Xl, yl = make_moons(n_samples=400, noise=0.3, random_state=1)
svm = LinearSVC(C=1.0).fit(Xl, yl)
lr = LogisticRegression(C=1.0).fit(Xl, yl)
unit = lambda w: w / np.linalg.norm(w)
print("boundary direction, linear SVM:     ", unit(svm.coef_[0]).round(3))
print("boundary direction, logistic regr.: ", unit(lr.coef_[0]).round(3))
print("agreement on predictions:", np.mean(svm.predict(Xl) == lr.predict(Xl)))

boundary direction, linear SVM:      [ 0.384 -0.923]
boundary direction, logistic regr.:  [ 0.384 -0.923]
agreement on predictions: 1.0


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [5]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
cols = ["dist_km", "age_yrs"]
svm = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1.0)).fit(df[cols], df["sold_fast"])
w = svm[-1].coef_[0]
print("margin width (standardised units):", round(2 / np.linalg.norm(w), 2))
print("support vectors: houses", (df["id"].iloc[svm[-1].support_]).tolist())

margin width (standardised units): 1.18
support vectors: houses [3, 7, 8, 2, 4]


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [6]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Margin width

Compute the margin of a linear SVM from its weight vector.

In [7]:
import numpy as np
w = np.array([3.0, 4.0])
margin = None  # ← your code here
check(margin, 0.4, hint='2 / ‖w‖.')

❌ Not yet. Hint: 2 / ‖w‖.


### Challenge 2: Hinge loss

Compute the average hinge loss for labels y (±1) and scores s.

In [8]:
import numpy as np
y = np.array([1, -1, 1, -1])
s = np.array([2.0, -0.5, 0.3, 1.0])
hinge = None  # ← your code here
check(hinge, 0.8, hint='max(0, 1 − y·s) for each point, then average.')

❌ Not yet. Hint: max(0, 1 − y·s) for each point, then average.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Margin width** — the missing line:

```python
margin = 2 / np.linalg.norm(w)
```

**2. Hinge loss** — the missing line:

```python
hinge = np.mean(np.maximum(0, 1 - y * s))
```